In [1]:
# Cell 1 - Imports
import os
import time
import joblib
import numpy as np
import pandas as pd
import fastf1

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    roc_auc_score,
)


In [2]:
# Cell 2 — Paths and cache config
BASE_DIR    = os.path.dirname(os.getcwd())           # goes up one level to F1/
DATA_PATH   = os.path.join(BASE_DIR, "data",   "f1_dataset_clean.csv")
MODEL_PATH  = os.path.join(BASE_DIR, "models", "model_v7.pkl")  
CACHE_PATH  = os.path.join(BASE_DIR, "cache")

os.makedirs(CACHE_PATH,                     exist_ok=True)
os.makedirs(os.path.dirname(DATA_PATH),     exist_ok=True)
os.makedirs(os.path.dirname(MODEL_PATH),    exist_ok=True)

fastf1.Cache.enable_cache(CACHE_PATH)
print("Base dir:", BASE_DIR)
print("Cache enabled at:", CACHE_PATH)

Base dir: d:\F1
Cache enabled at: d:\F1\cache


In [3]:
# Cell 3 — Track type lookup
TRACK_TYPE = {
    # Street circuits
    "Jeddah":        "street",
    "Baku":          "street",
    "Miami":         "street",
    "Monaco":        "street",
    "Marina Bay":    "street",
    "Las Vegas":     "street",
    "Melbourne":     "street",
    "Miami Gardens": "street",
    "Madrid":        "street",

    # Permanent circuits
    "Sakhir":            "permanent",
    "Barcelona":         "permanent",
    "Montréal":          "permanent",
    "Spielberg":         "permanent",
    "Silverstone":       "permanent",
    "Budapest":          "permanent",
    "Spa-Francorchamps": "permanent",
    "Zandvoort":         "permanent",
    "Monza":             "permanent",
    "Suzuka":            "permanent",
    "Lusail":            "permanent",
    "Austin":            "permanent",
    "Mexico City":       "permanent",
    "São Paulo":         "permanent",
    "Yas Island":        "permanent",
    "Shanghai":          "permanent",
    "Imola":             "permanent",
}

In [4]:
# Cell 4 — Fetch one race session to inspect
session_r = fastf1.get_session(2024, 1, "R")
session_r.load(laps=False, telemetry=False, weather=False, messages=False)

race_results = session_r.results[["FullName", "TeamName", "GridPosition", "Position", "Status"]].copy()
race_results["Year"]     = 2024
race_results["Round"]    = 1
race_results["Location"] = session_r.event["Location"]

print("Circuit:", session_r.event["Location"])
print(race_results.head(10))

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


Circuit: Sakhir
           FullName         TeamName  GridPosition  Position    Status  Year  \
1    Max Verstappen  Red Bull Racing           1.0       1.0  Finished  2024   
11     Sergio Perez  Red Bull Racing           5.0       2.0  Finished  2024   
55     Carlos Sainz          Ferrari           4.0       3.0  Finished  2024   
16  Charles Leclerc          Ferrari           2.0       4.0  Finished  2024   
63   George Russell         Mercedes           3.0       5.0  Finished  2024   
4      Lando Norris          McLaren           7.0       6.0  Finished  2024   
44   Lewis Hamilton         Mercedes           9.0       7.0  Finished  2024   
81    Oscar Piastri          McLaren           8.0       8.0  Finished  2024   
14  Fernando Alonso     Aston Martin           6.0       9.0  Finished  2024   
18     Lance Stroll     Aston Martin          12.0      10.0  Finished  2024   

    Round Location  
1       1   Sakhir  
11      1   Sakhir  
55      1   Sakhir  
16      1   Sakhir 

In [5]:
# Cell 5 — Fetch one qualifying session to inspect
session_q = fastf1.get_session(2024, 1, "Q")
session_q.load(laps=False, telemetry=False, weather=False, messages=False)

quali_results = session_q.results[["FullName", "Q1", "Q2", "Q3"]].copy()

# Each driver's best time is the minimum across Q1, Q2, Q3
# Some drivers don't make it to Q2/Q3 so those will be NaT (missing)
# .min(axis=1) ignores NaT automatically and picks the best available
quali_results["BestQualiTime"] = quali_results[["Q1", "Q2", "Q3"]].min(axis=1)
quali_results["BestQualiTime"] = quali_results["BestQualiTime"].dt.total_seconds()

quali_results = quali_results[["FullName", "BestQualiTime"]].copy()

print(quali_results.head(10))

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '55', '11', '14', '4', '81', '44', '27', '22', '18', '23', '3', '20', '77', '24', '2', '31', '10']


           FullName  BestQualiTime
1    Max Verstappen         89.179
16  Charles Leclerc         89.165
63   George Russell         89.485
55     Carlos Sainz         89.507
11     Sergio Perez         89.537
14  Fernando Alonso         89.542
4      Lando Norris         89.614
81    Oscar Piastri         89.683
44   Lewis Hamilton         89.710
27  Nico Hulkenberg         89.851


In [6]:
# Cell 6 — Merge race + quali into one round
df_round = race_results.merge(quali_results, on="FullName", how="left")

# Mark podium finishers as 1, everyone else 0
# This is our TARGET — what the model is trying to predict
df_round["Podium"] = (df_round["Position"] <= 3).astype(int)

# Map location to track type
df_round["TrackType"] = df_round["Location"].map(TRACK_TYPE)

print(df_round[["FullName", "TeamName", "GridPosition", "Position", "BestQualiTime", "Podium", "TrackType"]].head(10))

          FullName         TeamName  GridPosition  Position  BestQualiTime  \
0   Max Verstappen  Red Bull Racing           1.0       1.0         89.179   
1     Sergio Perez  Red Bull Racing           5.0       2.0         89.537   
2     Carlos Sainz          Ferrari           4.0       3.0         89.507   
3  Charles Leclerc          Ferrari           2.0       4.0         89.165   
4   George Russell         Mercedes           3.0       5.0         89.485   
5     Lando Norris          McLaren           7.0       6.0         89.614   
6   Lewis Hamilton         Mercedes           9.0       7.0         89.710   
7    Oscar Piastri          McLaren           8.0       8.0         89.683   
8  Fernando Alonso     Aston Martin           6.0       9.0         89.542   
9     Lance Stroll     Aston Martin          12.0      10.0         89.965   

   Podium  TrackType  
0       1  permanent  
1       1  permanent  
2       1  permanent  
3       0  permanent  
4       0  permanent  
5  

In [7]:
# Cell 7 — fetch_round function
def fetch_round(year, round_num):
    print(f"  Fetching {year} R{round_num}...")

    # --- Race ---
    try:
        session_r = fastf1.get_session(year, round_num, "R")
        session_r.load(laps=False, telemetry=False, weather=False, messages=False)
        race = session_r.results[["FullName", "TeamName", "GridPosition", "Position", "Status"]].copy()
        race["Year"]     = year
        race["Round"]    = round_num
        race["Location"] = session_r.event["Location"]
    except Exception as e:
        print(f"  ✗ Race failed: {e}")
        return None

    # --- Qualifying ---
    try:
        session_q = fastf1.get_session(year, round_num, "Q")
        session_q.load(laps=False, telemetry=False, weather=False, messages=False)
        q = session_q.results[["FullName", "Q1", "Q2", "Q3"]].copy()
        q["BestQualiTime"] = q[["Q1", "Q2", "Q3"]].min(axis=1).dt.total_seconds()
        q = q[["FullName", "BestQualiTime"]]
    except Exception as e:
        print(f"  ✗ Quali failed: {e}")
        q = pd.DataFrame(columns=["FullName", "BestQualiTime"])

    # --- Merge ---
    df = race.merge(q, on="FullName", how="left")
    df["Podium"]    = (df["Position"] <= 3).astype(int)
    df["TrackType"] = df["Location"].map(TRACK_TYPE)

    if df["TrackType"].isnull().any():
        unknown = df[df["TrackType"].isnull()]["Location"].unique()
        print(f"  ⚠ Unknown locations — add to TRACK_TYPE: {unknown}")

    print(f"  ✓ {year} R{round_num} — {len(df)} drivers")
    time.sleep(1)
    return df

In [8]:
# quick test
test = fetch_round(2024, 1)
print(test.shape)  # should be (20, 10)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  Fetching 2024 R1...


core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '55', '11', '14', '4', '81', '44', '27', '22', '18', '23', '3', '20', '77', '24', '2', '31', '10']


  ✓ 2024 R1 — 20 drivers
(20, 11)


In [11]:
# Cell 8 — Fetch all historical data (schedule-aware)
HISTORICAL_YEARS = [2023, 2024, 2025, 2026]  

frames = []

for year in HISTORICAL_YEARS:
    print(f"\n── {year} ──────────────────────")
    
    # Ask FastF1 how many rounds this year actually has
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    rounds = schedule["RoundNumber"].tolist()
    print(f"  {len(rounds)} rounds found")
    
    for round_num in rounds:
        df = fetch_round(year, round_num)
        if df is not None:
            frames.append(df)

df_raw = pd.concat(frames, ignore_index=True)

print(f"\nDone.")
print(f"Total rows : {len(df_raw)}")
print(f"Years      : {sorted(df_raw['Year'].unique())}")
print(f"Rounds     : {df_raw.groupby('Year')['Round'].nunique().to_dict()}")


── 2023 ──────────────────────


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '55', '14', '63', '44', '18', '31', '27', '4', '77', '24', '22', '23', '2', '20', '81', '21', '10']


  22 rounds found
  Fetching 2023 R1...
  ✓ 2023 R1 — 20 drivers


core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['11', '1', '14', '63', '44', '55', '16', '31', '10', '20', '22', '27', '24', '21', '81', '2', '4', '77', '23', '18']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['11', '16', '14', '63', '55', '18', '31', '44', '81', '10', '27', '24', '20', '77', '1', '22', '23', '21', '4', '2']


  Fetching 2023 R2...
  ✓ 2023 R2 — 20 drivers


...

  ✓ 2026 R21 — 0 drivers


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


  Fetching 2026 R22...


logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
core           INFO 	Finished loading data for 0 drivers: []
core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core        WARNING 	No result data for this session available on Erg

  ✓ 2026 R22 — 0 drivers

Done.
Total rows : 1486
Years      : [np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Rounds     : {2023: 22, 2024: 24, 2025: 24, 2026: 4}


In [12]:
# Cell 9 — Inspect raw data
print("Shape:", df_raw.shape)
print("\nData types:")
print(df_raw.dtypes)
print("\nMissing values:")
print(df_raw.isnull().sum())
print("\nSample rows:")
df_raw.head(5)

Shape: (1486, 11)

Data types:
FullName          object
TeamName          object
GridPosition     float64
Position         float64
Status            object
Year               int64
Round              int64
Location          object
BestQualiTime    float64
Podium             int64
TrackType         object
dtype: object

Missing values:
FullName          0
TeamName          0
GridPosition      1
Position          1
Status            0
Year              0
Round             0
Location          0
BestQualiTime    19
Podium            0
TrackType         0
dtype: int64

Sample rows:


,FullName,TeamName,GridPosition,Position,Status,Year,Round,Location,BestQualiTime,Podium,TrackType
0,Max Verstappen,Red Bull Racing,1.0,1.0,Finished,2023,1,Sakhir,89.708,1,permanent
1,Sergio Perez,Red Bull Racing,2.0,2.0,Finished,2023,1,Sakhir,89.846,1,permanent
2,Fernando Alonso,Aston Martin,5.0,3.0,Finished,2023,1,Sakhir,90.336,1,permanent
3,Carlos Sainz,Ferrari,4.0,4.0,Finished,2023,1,Sakhir,90.154,0,permanent
4,Lewis Hamilton,Mercedes,7.0,5.0,Finished,2023,1,Sakhir,90.384,0,permanent


In [13]:
# Cell 10 — Clean the data
def clean(df):
    df = df.copy()

    # Drop rows where we don't know the race result — can't train without the target
    df = df.dropna(subset=["Position"])

    # Cast to int now that NaNs are gone
    df["Position"]    = df["Position"].astype(int)
    df["GridPosition"] = df["GridPosition"].fillna(20).astype(int)

    # Impute missing quali times with worst time in that round + 5 second penalty
    worst_per_round = df.groupby(["Year", "Round"])["BestQualiTime"].transform("max")
    df["BestQualiTime"] = df["BestQualiTime"].fillna(worst_per_round + 5.0)

    # Sort chronologically — important for rolling features later
    df = df.sort_values(["Year", "Round", "Position"]).reset_index(drop=True)

    return df

df_clean = clean(df_raw)

print("Rows before:", len(df_raw))
print("Rows after :", len(df_clean))
print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

Rows before: 1486
Rows after : 1485

Missing values after cleaning:
FullName         0
TeamName         0
GridPosition     0
Position         0
Status           0
Year             0
Round            0
Location         0
BestQualiTime    0
Podium           0
TrackType        0
dtype: int64


In [24]:
# Cell 11 - Feature engineering (v6 enhanced, leakage-safe)
def other_driver_mean(df, by, column):
    counts = df.groupby(by)[column].transform("count")
    totals = df.groupby(by)[column].transform("sum")
    peer_mean = (totals - df[column]) / (counts - 1)
    return peer_mean.where(counts > 1, df[column])


def engineer_features(df):
    df = df.copy()

    # Qualifying features available before the race.
    round_key = ["Year", "Round"]
    team_round_key = ["Year", "Round", "TeamName"]

    df["QualiGapToPole"] = df.groupby(round_key)["BestQualiTime"].transform(
        lambda x: x - x.min()
    )
    df["QualiGapNormalized"] = df.groupby(round_key)["BestQualiTime"].transform(
        lambda x: (x - x.min()) / x.min() * 100
    )
    df["QualiGapToP3"] = df.groupby(round_key)["BestQualiTime"].transform(
        lambda x: x - x.nsmallest(min(3, len(x))).max()
    )
    df["QualiRankPct"] = df.groupby(round_key)["GridPosition"].rank(method="first", pct=True)
    df["GridPositionSquared"] = df["GridPosition"] ** 2
    df["MidfieldFlag"] = ((df["GridPosition"] >= 8) & (df["GridPosition"] <= 15)).astype(int)

    df["TeamGridAvg"] = df.groupby(team_round_key)["GridPosition"].transform("mean")
    df["TeamQualiGapAvg"] = df.groupby(team_round_key)["QualiGapNormalized"].transform("mean")
    teammate_grid = other_driver_mean(df, team_round_key, "GridPosition")
    teammate_quali = other_driver_mean(df, team_round_key, "BestQualiTime")
    df["TeammateGridDelta"] = df["GridPosition"] - teammate_grid
    df["TeammateQualiGap"] = df["BestQualiTime"] - teammate_quali

    # Historical driver race craft. Current race results are shifted out.
    df["PositionGain"] = df["GridPosition"] - df["Position"]
    df = df.sort_values(["FullName", "Year", "Round"])

    df["AvgPositionGainLast3"] = (
        df.groupby(["FullName", "Year"])["PositionGain"]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        .fillna(0.0)
    )
    df["FinishStdLast5"] = (
        df.groupby(["FullName", "Year"])["Position"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=2).std())
        .fillna(5.0)
    )
    df["AvgFinishLast3"] = (
        df.groupby(["FullName", "Year"])["Position"]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        .fillna(10.0)
    )
    df["PodiumRateLast5"] = (
        df.groupby(["FullName", "Year"])["Podium"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
        .fillna(0.15)
    )

    df["DNF"] = df["Status"].isin(["Retired", "Accident", "Collision damage", "Undertray", "Withdrew"]).astype(int)
    df["DNFRateLast5"] = (
        df.groupby(["FullName", "Year"])["DNF"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
        .fillna(0.1)
    )

    teammate_position = other_driver_mean(df, team_round_key, "Position")
    df["BeatTeammate"] = (df["Position"] < teammate_position).astype(int)
    df["BeatTeammateRate"] = (
        df.groupby(["FullName", "Year"])["BeatTeammate"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=2).mean())
        .fillna(0.5)
    )

    # Constructor form must be calculated once per team-race, shifted by race,
    # then merged back so both teammates receive the same pre-race value.
    team_round = (
        df.groupby(["TeamName", "Year", "Round"], as_index=False)
        .agg(
            TeamPodiumCurrent=("Podium", "mean"),
            TeamAvgFinishCurrent=("Position", "mean"),
        )
        .sort_values(["TeamName", "Year", "Round"])
    )
    team_round["ConstructorPodiumRate"] = (
        team_round.groupby(["TeamName", "Year"])["TeamPodiumCurrent"]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=2).mean())
        .fillna(0.1)
    )
    team_round["ConstructorAvgFinish"] = (
        team_round.groupby(["TeamName", "Year"])["TeamAvgFinishCurrent"]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        .fillna(10.0)
    )

    df = df.merge(
        team_round[["TeamName", "Year", "Round", "ConstructorPodiumRate", "ConstructorAvgFinish"]],
        on=["TeamName", "Year", "Round"],
        how="left",
    )

    df = df.sort_values(["Year", "Round", "GridPosition"]).reset_index(drop=True)
    df = df.drop(columns=["PositionGain", "DNF", "BeatTeammate"])

    return df


df_features = engineer_features(df_clean)

new_cols = [
    "QualiGapToPole", "QualiGapNormalized", "QualiGapToP3", "QualiRankPct",
    "GridPositionSquared", "MidfieldFlag", "TeamGridAvg", "TeamQualiGapAvg",
    "TeammateGridDelta", "TeammateQualiGap", "AvgPositionGainLast3",
    "FinishStdLast5", "DNFRateLast5", "BeatTeammateRate", "AvgFinishLast3",
    "PodiumRateLast5", "ConstructorPodiumRate", "ConstructorAvgFinish",
]
print("Missing values in engineered features:")
print(df_features[new_cols].isnull().sum())
print(f"\nTotal engineered features: {len(new_cols)}")


Missing values in engineered features:
QualiGapToPole           0
QualiGapNormalized       0
QualiGapToP3             0
QualiRankPct             0
GridPositionSquared      0
MidfieldFlag             0
TeamGridAvg              0
TeamQualiGapAvg          0
TeammateGridDelta        0
TeammateQualiGap         0
AvgPositionGainLast3     0
FinishStdLast5           0
DNFRateLast5             0
BeatTeammateRate         0
AvgFinishLast3           0
PodiumRateLast5          0
ConstructorPodiumRate    0
ConstructorAvgFinish     0
dtype: int64

Total engineered features: 18


In [25]:
# Cell 12 — Save featured dataset
df_features.to_csv(DATA_PATH, index=False)
print(f"Saved {len(df_features)} rows to {DATA_PATH}")
print(f"Columns: {df_features.columns.tolist()}")

Saved 1485 rows to d:\F1\data\f1_dataset_clean.csv
Columns: ['FullName', 'TeamName', 'GridPosition', 'Position', 'Status', 'Year', 'Round', 'Location', 'BestQualiTime', 'Podium', 'TrackType', 'QualiGapToPole', 'QualiGapNormalized', 'QualiGapToP3', 'QualiRankPct', 'GridPositionSquared', 'MidfieldFlag', 'TeamGridAvg', 'TeamQualiGapAvg', 'TeammateGridDelta', 'TeammateQualiGap', 'AvgPositionGainLast3', 'FinishStdLast5', 'AvgFinishLast3', 'PodiumRateLast5', 'DNFRateLast5', 'BeatTeammateRate', 'ConstructorPodiumRate', 'ConstructorAvgFinish']


In [26]:
# Cell 13 - Define feature columns and target
# These are the default production features. Extra qualifying/teammate features
# are engineered above as candidates, but kept out until backtests prove they
# beat the simpler grid/form model.
FEATURE_COLS = [
    # Qualifying/grid signal
    "GridPosition",
    "QualiGapNormalized",

    # Historical race craft
    "AvgPositionGainLast3",
    "FinishStdLast5",
    "DNFRateLast5",

    # Driver form
    "AvgFinishLast3",
    "PodiumRateLast5",
    "BeatTeammateRate",

    # Leakage-safe constructor form
    "ConstructorPodiumRate",
    "ConstructorAvgFinish",

    # Track type
    "TrackType_street",
    "TrackType_permanent",
]

CANDIDATE_FEATURE_COLS = [
    "QualiGapToP3",
    "QualiRankPct",
    "TeamGridAvg",
    "TeamQualiGapAvg",
    "TeammateGridDelta",
    "TeammateQualiGap",
    "GridPositionSquared",
    "MidfieldFlag",
]

TARGET = "Podium"

feature_groups = {
    "Qualifying/grid": 2,
    "Historical race craft": 3,
    "Driver form": 3,
    "Constructor form": 2,
    "Track": 2,
}

print(f"Default features: {len(FEATURE_COLS)}")
for label, count in feature_groups.items():
    print(f"  {label:22s}: {count:2d} ({count / len(FEATURE_COLS) * 100:.0f}%)")
print(f"Candidate features held out for experiments: {len(CANDIDATE_FEATURE_COLS)}")


Default features: 12
  Qualifying/grid       :  2 (17%)
  Historical race craft :  3 (25%)
  Driver form           :  3 (25%)
  Constructor form      :  2 (17%)
  Track                 :  2 (17%)
Candidate features held out for experiments: 8


In [28]:
# Cell 14 — One-hot encode and split
df_model = pd.get_dummies(df_features, columns=["TrackType"], dtype=int)

# Ensure both columns exist even if one track type is missing from a subset
for col in ["TrackType_street", "TrackType_permanent"]:
    if col not in df_model.columns:
        df_model[col] = 0

# Split: train on 2023+2024+2025, test on 2026
train_df = df_model[df_model["Year"] < 2026]
test_df  = df_model[df_model["Year"] == 2026]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET]

print(f"Train: {len(X_train)} rows ({train_df['Year'].nunique()} years)")
print(f"Test : {len(X_test)} rows (2026 only)")
print(f"\nPodium rate in train: {y_train.mean():.3f}")
print(f"Podium rate in test : {y_test.mean():.3f}")

Train: 1397 rows (3 years)
Test : 88 rows (2026 only)

Podium rate in train: 0.150
Podium rate in test : 0.136


In [29]:
# Cell 15 - Evaluation helpers and rolling backtest

def make_base_model():
    return GradientBoostingClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        min_samples_leaf=10,
        subsample=0.9,
        random_state=42,
    )


def top3_hits(eval_df, score_col, largest=True):
    correct, total = 0, 0
    per_round = []

    for (yr, rnd), grp in eval_df.groupby(["Year", "Round"]):
        ranked = grp.nlargest(3, score_col) if largest else grp.nsmallest(3, score_col)
        pred_top3 = set(ranked["FullName"])
        actual_top3 = set(grp[grp[TARGET] == 1]["FullName"])
        hits = len(pred_top3 & actual_top3)
        possible = min(3, len(actual_top3))
        correct += hits
        total += possible
        per_round.append({"Year": yr, "Round": rnd, "Hits": hits, "Possible": possible})

    return correct, total, correct / total if total else np.nan, pd.DataFrame(per_round)


def grid_top3_baseline(eval_df):
    return top3_hits(eval_df, "GridPosition", largest=False)


def evaluate_predictions(eval_df, y_true, proba, label):
    eval_df = eval_df.copy()
    eval_df["proba"] = proba
    pred = (proba >= 0.5).astype(int)

    print(f"-- {label} --")
    print(classification_report(y_true, pred, zero_division=0))

    metrics = {}
    if y_true.nunique() > 1:
        metrics["roc_auc"] = roc_auc_score(y_true, proba)
        metrics["average_precision"] = average_precision_score(y_true, proba)
        metrics["brier"] = brier_score_loss(y_true, proba)
        print(f"ROC AUC:           {metrics['roc_auc']:.4f}")
        print(f"Average Precision: {metrics['average_precision']:.4f}")
        print(f"Brier Score:       {metrics['brier']:.4f}")

    model_correct, model_total, model_rate, model_rounds = top3_hits(eval_df, "proba", largest=True)
    grid_correct, grid_total, grid_rate, grid_rounds = grid_top3_baseline(eval_df)

    print(f"Model Top-3:        {model_correct}/{model_total} ({model_rate:.1%})")
    print(f"Grid baseline Top-3:{grid_correct}/{grid_total} ({grid_rate:.1%})")

    metrics.update({
        "model_top3_correct": model_correct,
        "model_top3_total": model_total,
        "model_top3_rate": model_rate,
        "grid_top3_correct": grid_correct,
        "grid_top3_total": grid_total,
        "grid_top3_rate": grid_rate,
    })
    return metrics, model_rounds.merge(
        grid_rounds,
        on=["Year", "Round"],
        suffixes=("Model", "Grid"),
    )


def rolling_backtest(df_model, feature_cols):
    scenarios = [
        {"train_years": [2023], "test_year": 2024},
        {"train_years": [2023, 2024], "test_year": 2025},
        {"train_years": [2023, 2024, 2025], "test_year": 2026},
    ]
    rows = []

    for scenario in scenarios:
        scenario_train = df_model[df_model["Year"].isin(scenario["train_years"])]
        scenario_test = df_model[df_model["Year"] == scenario["test_year"]]
        if scenario_train.empty or scenario_test.empty or scenario_test[TARGET].nunique() < 2:
            continue

        weights = compute_sample_weights(scenario_train, max_year=max(scenario["train_years"]))
        backtest_model = CalibratedClassifierCV(make_base_model(), cv=3, method="isotonic")
        backtest_model.fit(
            scenario_train[feature_cols],
            scenario_train[TARGET],
            sample_weight=weights,
        )

        scored = scenario_test.copy()
        scored["proba"] = backtest_model.predict_proba(scenario_test[feature_cols])[:, 1]
        model_correct, model_total, model_rate, _ = top3_hits(scored, "proba", largest=True)
        grid_correct, grid_total, grid_rate, _ = grid_top3_baseline(scored)

        rows.append({
            "TrainYears": "+".join(str(y) for y in scenario["train_years"]),
            "TestYear": scenario["test_year"],
            "ROC_AUC": roc_auc_score(scenario_test[TARGET], scored["proba"]),
            "AveragePrecision": average_precision_score(scenario_test[TARGET], scored["proba"]),
            "Brier": brier_score_loss(scenario_test[TARGET], scored["proba"]),
            "ModelTop3": f"{model_correct}/{model_total}",
            "ModelTop3Rate": model_rate,
            "GridTop3": f"{grid_correct}/{grid_total}",
            "GridTop3Rate": grid_rate,
        })

    return pd.DataFrame(rows)


In [30]:
# Cell 16 - Era/sample weights
DECAY_FACTOR = 0.6


def compute_sample_weights(df, max_year):
    gap = max_year - df["Year"]
    weights = DECAY_FACTOR ** gap
    return weights.values


train_weights = compute_sample_weights(train_df, max_year=train_df["Year"].max())

all_weights = compute_sample_weights(df_model, max_year=2026)
for year in sorted(df_model["Year"].unique()):
    mask = df_model["Year"] == year
    print(f"{year}: display weight = {all_weights[mask][0]:.4f}  ({mask.sum()} rows)")

for year in sorted(train_df["Year"].unique()):
    mask = train_df["Year"] == year
    print(f"train {year}: weight = {train_weights[mask][0]:.4f}  ({mask.sum()} rows)")


2023: display weight = 0.2160  (439 rows)
2024: display weight = 0.3600  (479 rows)
2025: display weight = 0.6000  (479 rows)
2026: display weight = 1.0000  (88 rows)
train 2023: weight = 0.3600  (439 rows)
train 2024: weight = 0.6000  (479 rows)
train 2025: weight = 1.0000  (479 rows)


In [31]:
# Cell 17 - Train base model with era weights
base_model = make_base_model()
base_model.fit(X_train, y_train, sample_weight=train_weights)
print("Base model trained")


Base model trained


In [32]:
# Cell 18 - Probability calibration
model = CalibratedClassifierCV(make_base_model(), cv=3, method="isotonic")
model.fit(X_train, y_train, sample_weight=train_weights)
print("Calibrated model ready")


Calibrated model ready


In [ ]:
# Cell 19 - Evaluation
proba = model.predict_proba(X_test)[:, 1]
metrics_2026, per_round_2026 = evaluate_predictions(test_df, y_test, proba, "2026 holdout")

print("\nPer-round Top-3 hits:")
print(per_round_2026)

print("\nRolling backtest:")
rolling_results = rolling_backtest(df_model, FEATURE_COLS)
print(rolling_results)


-- 2026 holdout --
              precision    recall  f1-score   support

           0       0.92      0.96      0.94        76
           1       0.67      0.50      0.57        12

    accuracy                           0.90        88
   macro avg       0.80      0.73      0.76        88
weighted avg       0.89      0.90      0.89        88

ROC AUC:           0.9457
Average Precision: 0.7329
Brier Score:       0.0660
Model Top-3:        9/12 (75.0%)
Grid baseline Top-3:8/12 (66.7%)

Per-round Top-3 hits:
   Year  Round  HitsModel  PossibleModel  HitsGrid  PossibleGrid
0  2026      1          3              3         2             3
1  2026      2          3              3         3             3
2  2026      3          2              3         2             3
3  2026      4          1              3         1             3

Rolling backtest:
       TrainYears  TestYear   ROC_AUC  AveragePrecision     Brier ModelTop3  \
0            2023      2024  0.920745          0.593910  0.08227

In [ ]:
# Cell 20 — Feature importance
print("-- Feature Importance --")
for feat, imp in sorted(
    zip(FEATURE_COLS, base_model.feature_importances_), key=lambda x: -x[1]
):
    print(f"  {feat:25s} {imp:.4f}")


-- Feature Importance --
  GridPosition              0.5446
  QualiGapNormalized        0.1103
  FinishStdLast5            0.0676
  ConstructorAvgFinish      0.0624
  AvgPositionGainLast3      0.0558
  AvgFinishLast3            0.0510
  ConstructorPodiumRate     0.0449
  PodiumRateLast5           0.0313
  BeatTeammateRate          0.0217
  DNFRateLast5              0.0044
  TrackType_permanent       0.0041
  TrackType_street          0.0017


In [ ]:
# Cell 21 - Defer model save until v7 feature selection
print("Model save deferred until SELECTED_FEATURE_COLS_V7 is chosen in Cell 22.")


In [ ]:
# Cell 22 - v7 feature-set experiments
TOP3_PERM_REPEATS = 30


def unique_features(features):
    return list(dict.fromkeys(features))


FEATURE_EXPERIMENTS = {
    "v6_baseline": FEATURE_COLS,
    "plus_midfield": FEATURE_COLS + ["MidfieldFlag"],
    "plus_grid_squared": FEATURE_COLS + ["GridPositionSquared"],
    "plus_quali_rank_pct": FEATURE_COLS + ["QualiRankPct"],
    "plus_team_quali_gap_avg": FEATURE_COLS + ["TeamQualiGapAvg"],
    "plus_midfield_grid_squared": FEATURE_COLS + ["MidfieldFlag", "GridPositionSquared"],
    "plus_midfield_quali_rank": FEATURE_COLS + ["MidfieldFlag", "QualiRankPct"],
    "plus_grid_squared_quali_rank": FEATURE_COLS + ["GridPositionSquared", "QualiRankPct"],
    "plus_all_positive": FEATURE_COLS + [
        "MidfieldFlag",
        "GridPositionSquared",
        "QualiRankPct",
        "TeamQualiGapAvg",
    ],
    "plus_all_candidates": FEATURE_COLS + CANDIDATE_FEATURE_COLS,
}
FEATURE_EXPERIMENTS = {
    name: unique_features(cols) for name, cols in FEATURE_EXPERIMENTS.items()
}


def evaluate_feature_experiments(df_model, experiments):
    rows = []
    fold_rows = []

    baseline_bt = rolling_backtest(df_model, experiments["v6_baseline"])
    baseline_top3 = baseline_bt["ModelTop3Rate"].mean()
    baseline_brier = baseline_bt["Brier"].mean()

    for name, cols in experiments.items():
        bt = rolling_backtest(df_model, cols)
        mean_top3 = bt["ModelTop3Rate"].mean()
        mean_grid = bt["GridTop3Rate"].mean()
        mean_brier = bt["Brier"].mean()
        mean_auc = bt["ROC_AUC"].mean()
        mean_ap = bt["AveragePrecision"].mean()
        folds_under_grid = int((bt["ModelTop3Rate"] < bt["GridTop3Rate"]).sum())

        rows.append({
            "Experiment": name,
            "FeatureCount": len(cols),
            "MeanTop3Rate": mean_top3,
            "DeltaVsBaseline": mean_top3 - baseline_top3,
            "MeanGridTop3Rate": mean_grid,
            "DeltaVsGrid": mean_top3 - mean_grid,
            "MeanROC_AUC": mean_auc,
            "MeanAveragePrecision": mean_ap,
            "MeanBrier": mean_brier,
            "DeltaBrier": mean_brier - baseline_brier,
            "FoldsUnderGrid": folds_under_grid,
        })

        bt_detail = bt.copy()
        bt_detail.insert(0, "Experiment", name)
        fold_rows.append(bt_detail)

    summary = pd.DataFrame(rows).sort_values(
        ["MeanTop3Rate", "MeanBrier"], ascending=[False, True]
    )
    fold_detail = pd.concat(fold_rows, ignore_index=True)
    return summary, fold_detail


def train_fold_model(fold_train, feature_cols):
    weights = compute_sample_weights(fold_train, max_year=fold_train["Year"].max())
    fold_model = CalibratedClassifierCV(make_base_model(), cv=3, method="isotonic")
    fold_model.fit(fold_train[feature_cols], fold_train[TARGET], sample_weight=weights)
    return fold_model


def top3_permutation_importance(df_model, feature_cols, candidate, n_repeats=30, random_state=42):
    rng = np.random.default_rng(random_state)
    scenarios = [
        {"train_years": [2023], "test_year": 2024},
        {"train_years": [2023, 2024], "test_year": 2025},
        {"train_years": [2023, 2024, 2025], "test_year": 2026},
    ]
    drops = []

    for scenario in scenarios:
        fold_train = df_model[df_model["Year"].isin(scenario["train_years"])]
        fold_test = df_model[df_model["Year"] == scenario["test_year"]].copy()
        if fold_train.empty or fold_test.empty or fold_test[TARGET].nunique() < 2:
            continue

        fold_model = train_fold_model(fold_train, feature_cols)
        original = fold_test.copy()
        original["proba"] = fold_model.predict_proba(original[feature_cols])[:, 1]
        _, _, original_rate, _ = top3_hits(original, "proba", largest=True)

        for _ in range(n_repeats):
            permuted = fold_test.copy()
            permuted[candidate] = rng.permutation(permuted[candidate].to_numpy())
            permuted["proba"] = fold_model.predict_proba(permuted[feature_cols])[:, 1]
            _, _, permuted_rate, _ = top3_hits(permuted, "proba", largest=True)
            drops.append(original_rate - permuted_rate)

    return float(np.mean(drops)), float(np.std(drops))


print("=" * 76)
print("  v7 feature-set experiment summary")
print("=" * 76)
experiment_summary, experiment_folds = evaluate_feature_experiments(df_model, FEATURE_EXPERIMENTS)
print(
    experiment_summary[
        [
            "Experiment",
            "FeatureCount",
            "MeanTop3Rate",
            "DeltaVsBaseline",
            "MeanGridTop3Rate",
            "DeltaVsGrid",
            "MeanBrier",
            "FoldsUnderGrid",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 76)
print("  Fold detail for top experiments")
print("=" * 76)
top_experiments = experiment_summary.head(4)["Experiment"].tolist()
print(
    experiment_folds[experiment_folds["Experiment"].isin(top_experiments)][
        [
            "Experiment",
            "TrainYears",
            "TestYear",
            "ModelTop3",
            "ModelTop3Rate",
            "GridTop3",
            "GridTop3Rate",
            "Brier",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 76)
print("  Top-3 permutation importance for added features in best non-baseline set")
print("=" * 76)
best_non_baseline = experiment_summary[
    experiment_summary["Experiment"] != "v6_baseline"
].iloc[0]
best_experiment_name = best_non_baseline["Experiment"]
best_feature_cols = FEATURE_EXPERIMENTS[best_experiment_name]
added_features = [feat for feat in best_feature_cols if feat not in FEATURE_COLS]

print(f"Best non-baseline experiment: {best_experiment_name}")
if not added_features:
    print("No added features to permute.")
else:
    permutation_rows = []
    for feature in added_features:
        mean_drop, std_drop = top3_permutation_importance(
            df_model,
            best_feature_cols,
            feature,
            n_repeats=TOP3_PERM_REPEATS,
        )
        permutation_rows.append({
            "Feature": feature,
            "MeanTop3Drop": mean_drop,
            "StdTop3Drop": std_drop,
        })

    top3_permutation_summary = pd.DataFrame(permutation_rows).sort_values(
        "MeanTop3Drop", ascending=False
    )
    print(top3_permutation_summary.to_string(index=False))

print("\n" + "=" * 76)
print("  Recommended v7 feature set")
print("=" * 76)
selected_experiment = experiment_summary.iloc[0]["Experiment"]
SELECTED_FEATURE_COLS_V7 = FEATURE_EXPERIMENTS[selected_experiment]
print(f"Selected experiment: {selected_experiment}")
print(f"Selected feature count: {len(SELECTED_FEATURE_COLS_V7)}")
print(SELECTED_FEATURE_COLS_V7)


In [ ]:
# Cell 23 - Train and save selected v7 model
X_train_v7 = train_df[SELECTED_FEATURE_COLS_V7]
X_test_v7 = test_df[SELECTED_FEATURE_COLS_V7]

selected_model = CalibratedClassifierCV(make_base_model(), cv=3, method="isotonic")
selected_model.fit(X_train_v7, y_train, sample_weight=train_weights)

selected_proba = selected_model.predict_proba(X_test_v7)[:, 1]
selected_metrics_2026, selected_per_round_2026 = evaluate_predictions(
    test_df,
    y_test,
    selected_proba,
    f"2026 holdout - {selected_experiment}",
)

print("\nSelected v7 rolling backtest:")
selected_rolling_results = rolling_backtest(df_model, SELECTED_FEATURE_COLS_V7)
print(selected_rolling_results)

joblib.dump(selected_model, MODEL_PATH)
print(f"Model saved -> {MODEL_PATH}")
